# Deploy and Run the Circuit Function

This interactive guide shows how to upload the circuit function to Qiskit Serverless and run two example workloads: a multi-observable survey across a sweep of parameter sets, and a direct comparison of error mitigation strategies on the same circuit.

### Requirements

This guide was developed with the following local package versions:

In [120]:
from qiskit import __version__ as qiskit_version
from qiskit_ibm_catalog import __version__ as catalog_version
import numpy as np

print("qiskit version:", qiskit_version)
print("qiskit-ibm-catalog version:", catalog_version)
print("numpy version:", np.__version__)

qiskit version: 2.2.3
qiskit-ibm-catalog version: 0.16.0
numpy version: 2.5.0


## 1. Authentication

Use `qiskit-ibm-catalog` to authenticate to `QiskitServerless` with your API key (token) and CRN (instance), which you can find on the [IBM Quantum Platform](https://quantum.cloud.ibm.com) dashboard. This will allow you to locally instantiate the serverless client to upload or run the selected function:

```python
from qiskit_ibm_catalog import QiskitServerless
serverless = QiskitServerless(channel="ibm_quantum_platform", token="MY_TOKEN", instance="MY_CRN")
```

You can optionally use `save_account()` to save your credentials in your local environment (see the [Set up your IBM Cloud account](/docs/guides/cloud-setup#cloud-save) guide). Note that this writes your credentials to the same file as [`QiskitRuntimeService.save_account()`](/docs/api/qiskit-ibm-runtime/qiskit-runtime-service#save_account):

```python
QiskitServerless.save_account(channel="ibm_quantum_platform", token="MY_TOKEN", instance="MY_CRN")
```

If the account is saved, there is no need to provide the token to authenticate:

In [ ]:
from qiskit_ibm_catalog import QiskitServerless

# Authenticate to the remote cluster
# In this case, loading a named saved account
# serverless = QiskitServerless(name="my_account")

# REPLACE WITH YOUR OWN CREDENTIALS or SAVED ACCOUNT
serverless = QiskitServerless(channel="ibm_quantum_platform", token="MY_TOKEN", instance="MY_CRN")

## 2. Upload the custom function

To upload a Qiskit Function, you must first instantiate a `QiskitFunction` object that defines the function source code. The title will allow you to identify the function once it's in the remote cluster. The main entry point is the file that contains `if __name__ == "__main__"`. The `working_dir` contains the entrypoint together with the `options/` sub-package it imports at runtime.

In [122]:
from qiskit_ibm_catalog import QiskitFunction

template = QiskitFunction(
    title="circuit_function_template",
    entrypoint="circuit_function_entrypoint.py",
    working_dir="./source_files/",  # all files in this directory will be uploaded
)
print(template)

QiskitFunction(circuit_function_template)


Once the instance is ready, upload it to serverless:

In [123]:
serverless.upload(template)

QiskitFunction(circuit_function_template)

To check if the program successfully uploaded, use `serverless.list()`:

In [124]:
serverless.list()

[QiskitFunction(circuit_function_template)]

## 3. Load and run the custom function remotely

The function template has been uploaded, so you can run it remotely with Qiskit Serverless. First, load the template by name:

In [125]:
template = serverless.load("circuit_function_template")
print(template)

QiskitFunction(circuit_function_template)


### Example 1 — Multi-observable survey on a parameterized ansatz

This example measures five two-qubit observables on a 4-qubit `RealAmplitudes` ansatz across a sweep of 5 random parameter sets. Each observable is paired with the full parameter sweep as its own PUB, so a single `run()` call returns five `PubResult`s, each with `evs` of shape `(5,)` — one expectation value per parameter set.

| Observable | Physical meaning |
|---|---|
| `IIZZ` | ZZ correlation, qubits 0–1 |
| `ZZII` | ZZ correlation, qubits 2–3 |
| `IIXX` | XX correlation, qubits 0–1 |
| `IIYY` | YY correlation, qubits 0–1 |
| `IZZI` | ZZ correlation across the inner bond, qubits 1–2 |

In [126]:
import numpy as np
from qiskit.circuit.library import real_amplitudes

NUM_QUBITS = 4
NUM_PARAM_SETS = 5

ansatz = real_amplitudes(num_qubits=NUM_QUBITS, reps=1)  # 8 parameters

rng = np.random.default_rng(seed=42)
param_sets = rng.uniform(0, 2 * np.pi, size=(NUM_PARAM_SETS, ansatz.num_parameters))

# One PUB per observable — each pairs the same ansatz and parameter sweep
# with a single Pauli string. obs.shape=() broadcasts against params.shape=(5,),
# so each pub.shape=(5,) and evs.shape=(5,) after execution.
obs_labels = ["IIZZ", "ZZII", "IIXX", "IIYY", "IZZI"]
pubs_survey = [(ansatz, obs, param_sets) for obs in obs_labels]

print("num_parameters:", ansatz.num_parameters)
print("num_param_sets:", NUM_PARAM_SETS)

num_parameters: 8
num_param_sets: 5


Submit one job carrying all five PUBs. Each PUB pairs the same ansatz and parameter sweep with one observable string:

In [127]:
job_survey = template.run(
    backend_name="ibm_kingston",
    pubs=pubs_survey,
)
print("job_id:", job_survey.job_id)

job_id: ed636a1c-8710-4ac8-b00f-4519fdf150dd


Poll until the job reaches a terminal state:

In [129]:
import time

t0 = time.time()
while True:
    status = job_survey.status()
    print(f"time = {time.time()-t0:.2f}, status = {status}")
    if status in ("DONE", "ERROR", "CANCELED"):
        break
    time.sleep(5)

time = 0.87, status = QUEUED
time = 8.45, status = QUEUED
time = 15.72, status = QUEUED
time = 21.56, status = QUEUED
time = 28.37, status = QUEUED
time = 34.22, status = QUEUED
time = 39.99, status = QUEUED
time = 47.66, status = QUEUED
time = 53.61, status = QUEUED
time = 59.42, status = QUEUED
time = 65.15, status = QUEUED
time = 72.40, status = QUEUED
time = 79.52, status = QUEUED
time = 87.30, status = QUEUED
time = 93.24, status = QUEUED
time = 99.04, status = QUEUED
time = 104.79, status = QUEUED
time = 112.49, status = QUEUED
time = 120.48, status = QUEUED
time = 127.75, status = QUEUED
time = 133.58, status = QUEUED
time = 140.45, status = QUEUED
time = 147.88, status = QUEUED
time = 155.23, status = QUEUED
time = 161.03, status = QUEUED
time = 166.77, status = QUEUED
time = 172.60, status = QUEUED
time = 178.33, status = QUEUED
time = 184.12, status = QUEUED
time = 189.91, status = QUEUED
time = 196.97, status = QUEUED
time = 202.81, status = QUEUED
time = 208.54, status = QU

Retrieve the results. There are five `PubResult`s — one per observable. Each `data.evs` has shape `(5,)` — one expectation value per parameter set. Print the full table to see how each observable varies across the random sweep:

In [130]:
result_survey = job_survey.result()
hw = result_survey["hw_results"]  # 5 PubResults, one per observable

col_header = "".join(f"  θ-set {i}" for i in range(NUM_PARAM_SETS))
print(f"{'':10s}{col_header}")
for label, pr in zip(obs_labels, hw):
    vals = "".join(f"  {v:>+7.3f}" for v in pr.data.evs)
    print(f"{label:10s}{vals}")

            θ-set 0  θ-set 1  θ-set 2  θ-set 3  θ-set 4
IIZZ         -0.654   +0.766   +0.284   +0.652   +0.600
ZZII         -0.728   +0.079   -0.747   +0.011   -0.055
IIXX         -0.706   -0.828   -0.876   +0.022   -0.671
IIYY         -0.916   +0.676   +0.317   +0.333   +0.621
IZZI         -0.011   +0.072   +0.237   +0.824   +0.112


Check the function logs:

In [131]:
print(job_survey.logs())

2026-07-08 17:43:49,476	INFO job_manager.py:587 -- Runtime env is setting up.
Running entrypoint for job raysubmit_au5zBqUcHmdApL62: python circuit_function_entrypoint.py
INFO: Input options are: None
INFO: Backend used: ibm_kingston
INFO: Instance used: None
INFO: Starting runtime service
qiskit_runtime_service._discover_account:WARNING:2026-07-08 17:43:52,562: Loading account with the given token. A saved account will not be used.
INFO: Backend: ibm_kingston
INFO: Qiskit runtime job d978onqf47jc73a6f3q0



The result dict also carries a `"metadata"` key with per-stage CPU timing:

In [ ]:
result_survey["metadata"]

### Example 2 — Error mitigation comparison

Real quantum hardware introduces systematic errors: decoherence, gate noise, and measurement crosstalk. The circuit function exposes a structured `options` dict whose keys map directly onto the IBM Runtime `EstimatorV2` options. This example uses the same 4-qubit `RealAmplitudes` ansatz but fixes a single parameter set so that differences between mitigation strategies are directly comparable.

Two jobs are submitted back-to-back:

| Job | `mitigation_level` | Extra options | What this exercises |
|---|---|---|---|
| `job_l1` | `1` (default) | — | DD + measurement twirling (TREX) only |
| `job_l2` | `2` | `XY4` DD sequence, custom ZNE noise factors | Level-1 stack + gate twirling + ZNE via gate-folding with explicit noise schedule |

`mitigation_level=2` enables gate twirling and Zero-Noise Extrapolation (ZNE). The `resilience.zne.noise_factors` key overrides the default ZNE schedule `[1, 3, 5]` with a finer `[1, 2, 3, 4]` — four noise amplification points instead of three, which gives the linear extrapolator more signal at the cost of additional circuit executions. The DD sequence is also changed from the default (`XX`) to `XY4`, which suppresses a wider class of systematic pulse errors on superconducting hardware.

> **Local testing note:** in `channel="local"` mode the Estimator is a simulator, so DD, twirling, and ZNE have no numerical effect. The options are still validated and accepted — which is what the function tests. The quantitative benefit only manifests on real or staging backends.

In [132]:
import numpy as np
from qiskit.circuit.library import real_amplitudes
from qiskit.quantum_info import SparsePauliOp

NUM_QUBITS = 4

ansatz = real_amplitudes(num_qubits=NUM_QUBITS, reps=2)  # 12 parameters
observable = SparsePauliOp("IIZZ")  # ZZ correlation on qubits 0,1

# Fix a single parameter set so the two jobs are directly comparable.
rng = np.random.default_rng(seed=7)
params = rng.uniform(0, 2 * np.pi, size=(ansatz.num_parameters,))

# Level 1 — DD + measurement twirling (TREX). This is the default and requires
# no extra options, but we set it explicitly to make the comparison clear.
options_l1 = {
    "mitigation_level": 1,
    "default_precision": 0.02,
}

# Level 2 — adds gate twirling and ZNE via gate-folding. We override two
# sub-options to make this job more interesting:
#   • dynamical_decoupling.sequence_type = "XY4" — a 4-pulse DD sequence that
#     suppresses both X- and Y-axis systematic errors, compared to the default
#     2-pulse "XX" sequence.
#   • resilience.zne.noise_factors = [1, 2, 3, 4] — four amplification points
#     instead of the default three, giving the linear and exponential extrapolators
#     an additional data point for a more accurate zero-noise estimate.
options_l2 = {
    "mitigation_level": 2,
    "default_precision": 0.02,
    "dynamical_decoupling": {
        "sequence_type": "XY4",
    },
    "resilience": {
        "zne": {
            "noise_factors": [1, 2, 3, 4],
            "extrapolator": ["exponential", "linear"],
        }
    },
}

Submit both jobs. Launching them before either has finished means both queue and execute in parallel on the backend:

In [133]:
BACKEND = "ibm_kingston"

job_l1 = template.run(
    backend_name=BACKEND,
    pubs=[(ansatz, observable, params)],
    options=options_l1,
)
job_l2 = template.run(
    backend_name=BACKEND,
    pubs=[(ansatz, observable, params)],
    options=options_l2,
)

print("job_l1 id:", job_l1.job_id)
print("job_l2 id:", job_l2.job_id)

job_l1 id: 5f0d702c-8fb4-4bfa-8b34-385caae3f69d
job_l2 id: c4be104c-cc33-43e5-b607-8dc744dfa7c9


Poll both jobs to completion. Each is polled in a round-robin loop so neither blocks the other:

In [134]:
import time

terminal = {"DONE", "ERROR", "CANCELED"}
t0 = time.time()

while True:
    s1 = job_l1.status()
    s2 = job_l2.status()
    print(f"time = {time.time()-t0:5.2f}  job_l1={s1:<40s}  job_l2={s2}")
    if s1 in terminal and s2 in terminal:
        break
    time.sleep(2)

time =  4.08  job_l1=QUEUED                                    job_l2=QUEUED
time =  7.66  job_l1=QUEUED                                    job_l2=QUEUED
time = 14.01  job_l1=QUEUED                                    job_l2=QUEUED
time = 17.45  job_l1=QUEUED                                    job_l2=QUEUED
time = 24.16  job_l1=QUEUED                                    job_l2=QUEUED
time = 27.91  job_l1=QUEUED                                    job_l2=QUEUED
time = 33.68  job_l1=QUEUED                                    job_l2=QUEUED
time = 39.32  job_l1=QUEUED                                    job_l2=QUEUED
time = 42.65  job_l1=QUEUED                                    job_l2=QUEUED
time = 46.30  job_l1=QUEUED                                    job_l2=QUEUED
time = 49.95  job_l1=RUNNING                                   job_l2=QUEUED
time = 53.42  job_l1=RUNNING                                   job_l2=QUEUED
time = 58.47  job_l1=RUNNING                                   job_l2=QUEUED

Retrieve results from both jobs and compare the expectation value and standard deviation side by side. A lower `std` at a comparable `evs` indicates the higher mitigation level is recovering more signal from the noisy backend without biasing the estimate:

In [135]:
result_l1 = job_l1.result()
result_l2 = job_l2.result()

pr_l1 = result_l1["hw_results"][0]
pr_l2 = result_l2["hw_results"][0]

evs_l1 = float(pr_l1.data.evs)
stds_l1 = float(pr_l1.data.stds)
evs_l2 = float(pr_l2.data.evs)
stds_l2 = float(pr_l2.data.stds)

print("Observable : IIZZ")
print("━" * 48)
print(f"{'':14s} {'evs':>10s}  {'std':>10s}")
print(f"{'Level 1':14s} {evs_l1:+.4f}  {stds_l1:10.4f}")
print(f"{'Level 2':14s} {evs_l2:+.4f}  {stds_l2:10.4f}")
print("━" * 48)
delta_evs = evs_l2 - evs_l1
delta_stds = stds_l2 - stds_l1
direction = "lower std with level 2" if delta_stds < 0 else "higher std with level 2"
print(f"Δ evs  (L2 − L1) :  {delta_evs:+.4f}")
print(f"Δ std  (L2 − L1) :  {delta_stds:+.4f}  ({direction})")

Observable : IIZZ
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                      evs         std
Level 1        -0.0859      0.0161
Level 2        -0.1523      0.0454
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Δ evs  (L2 − L1) :  -0.0664
Δ std  (L2 − L1) :  +0.0293  (higher std with level 2)


You can also retrieve the resource metadata for each job to understand the extra QPU time cost of the more aggressive mitigation strategy. Level 2 runs more circuit variants (for ZNE amplification and gate twirling randomizations), so `EXECUTING_QPU` CPU time will be proportionally higher:

In [136]:
stages = [
    "RUNNING: OPTIMIZING_FOR_HARDWARE",
    "RUNNING: WAITING_FOR_QPU",
    "RUNNING: EXECUTING_QPU",
    "RUNNING: POST_PROCESSING",
]

meta_l1 = result_l1["metadata"]["resources_usage"]
meta_l2 = result_l2["metadata"]["resources_usage"]

print(f"{'Stage':<38s} {'Level 1 (s)':>12s}  {'Level 2 (s)':>12s}")
print("─" * 60)
for stage in stages:
    t1 = meta_l1.get(stage, {}).get("CPU_TIME", float("nan"))
    t2 = meta_l2.get(stage, {}).get("CPU_TIME", float("nan"))
    print(f"{stage:<38s} {t1:>12.4f}  {t2:>12.4f}")

Stage                                   Level 1 (s)   Level 2 (s)
────────────────────────────────────────────────────────────
RUNNING: OPTIMIZING_FOR_HARDWARE             0.7531        0.6964
RUNNING: WAITING_FOR_QPU                     7.4953        3.8348
RUNNING: EXECUTING_QPU                      75.7927       26.3303
RUNNING: POST_PROCESSING                     1.9467        0.7112
